# City Tuning
This notebook will investigate the following:
1. City Creation - what city environments will be used.
2. Genetic Algorithm Parameter Tuning - what parameters should be used in the algorithm.
3. Agent Selection - how many agents should be used in the simulations.

In [36]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [37]:
import numpy as np

from utils.environment import Environment 

## 1 City Creation
For the dissertation I want to compare the performance of GA and greedy across different situation, I need to create different cities which represent different severity of bottle necks.

The following 4 cities need to be created.

| City Name | Bottleneck Corridoors | Open Corridoors | Expected PoA |
| --------- | --------------------- | --------------- | ------------ |
| Connected | 3 | 3| Low |
| Moderate | 2 | 3| Medium |
| Severe | 1 | 3| High |
| Extreme | 1 | 1| Very High |


I will use the betweeness score of the city to understand the bottleneck severity:
* **Max betweenness < 0.1**  → well connected, no dominant bottleneck
* **Max betweenness 0.1-0.3** → moderate bottleneck, some critical junctions
* **Max betweenness > 0.3**  → severe bottleneck, network heavily dependent on few nodes

### Connected City

In [38]:
SEED = 43


In [50]:
from city_creation import *

num_bottleneck = 3
num_open = 3

G_connected, start_c, exit_c = create_city(
    seed=SEED,
    bottleneck_corridors=num_bottleneck,
    open_corridors=num_open,
    exit_connections=3,
    start_connections=3,
)

betweeness_c = bottleneck_score_betweeness(G_connected)
avg_c = bottleneck_score_avg_path(G_connected, start_c, exit_c)
cuts_c = bottleneck_score_min_cut(G_connected, start_c, exit_c)

visualise_city(G_connected, title='Connected City', save_path='images/connected_city.png')

pickle.dump(
    {"graph": G_connected, "starts": start_c, "exits": exit_c},
    open(os.path.join("graphs/connected_city.pickle"), "wb"),
)

### Moderate City

In [51]:
num_bottleneck = 2
num_open = 3

G_moderate, start_m, exit_m = create_city(
    seed=SEED,
    bottleneck_corridors=num_bottleneck,
    open_corridors=num_open,
    exit_connections=3,
    start_connections=3,
)

betweeness_m = bottleneck_score_betweeness(G_moderate)
avg_m = bottleneck_score_avg_path(G_moderate, start_m, exit_m)
cuts_m = bottleneck_score_min_cut(G_moderate, start_m, exit_m)
visualise_city(G_moderate, title='Moderate City', save_path='images/moderate_city.png')

pickle.dump(
    {"graph": G_moderate, "starts": start_m, "exits": exit_m},
    open(os.path.join("graphs/moderate_city.pickle"), "wb"),
)

### Severe City

In [52]:
num_bottleneck = 1
num_open = 3

G_severe, start_s, exit_s = create_city(
    seed=SEED,
    bottleneck_corridors=num_bottleneck,
    open_corridors=num_open,
    exit_connections=3,
    start_connections=3,
)

betweeness_s = bottleneck_score_betweeness(G_severe)
avg_s = bottleneck_score_avg_path(G_severe, start_s, exit_s)
cuts_s = bottleneck_score_min_cut(G_severe, start_s, exit_s)
visualise_city(G_severe, title='Severe City', save_path='images/severe_city.png')

pickle.dump(
    {"graph": G_severe, "starts": start_s, "exits": exit_s},
    open(os.path.join("graphs/severe_city.pickle"), "wb"),
)

### Extreme City

In [53]:
num_bottleneck = 1
num_open = 1

G_extreme, start_e, exit_e = create_city(
    seed=SEED,
    bottleneck_corridors=num_bottleneck,
    open_corridors=num_open,
    exit_connections=3,
    start_connections=3,
)

betweeness_e = bottleneck_score_betweeness(G_extreme)
avg_e = bottleneck_score_avg_path(G_extreme, start_e, exit_e)
cuts_e = bottleneck_score_min_cut(G_extreme, start_e, exit_e)
visualise_city(G_extreme, title='Extreme City', save_path='images/extreme_city.png')

pickle.dump(
    {"graph": G_extreme, "starts": start_e, "exits": exit_e},
    open(os.path.join("graphs/extreme_city.pickle"), "wb"),
)

### Summary Bottleneck Score

In [56]:
import pandas as pd 

cities = [
    ('connected', betweeness_c, avg_c, cuts_c),
    ('moderate', betweeness_m, avg_m, cuts_m),
    ('severe', betweeness_s, avg_s, cuts_s),
    ('extreme', betweeness_e, avg_e, cuts_e)
]


output = []
for city_name, between, avg, cuts in cities:
    output.append([
        city_name,
        between['mean'], between['max'],
        avg['mean_node_connectivity'], avg['min_node_connectivity'],
        cuts['mean_cut'], cuts['min_cut']
    ])
    
output_df = pd.DataFrame(output, columns=[
    'City Name', 'Betweeness Mean', 'Betweeness Max', 'Mean Node Connectivity', 'Min Node Connectivity', 'Mean Cuts', 'Min Cuts']
)
output_df


,City Name,Betweeness Mean,Betweeness Max,Mean Node Connectivity,Min Node Connectivity,Mean Cuts,Min Cuts
0,connected,0.060513,0.290137,3.0,3,3.0,3
1,moderate,0.061026,0.257804,3.0,3,3.0,3
2,severe,0.063205,0.295751,3.0,3,3.0,3
3,extreme,0.092692,0.466206,2.0,2,2.0,2


### Node Count across cities

In [44]:
graph_cities = [
    ('connected', G_connected, start_c, exit_c), 
    ('moderate', G_moderate, start_m, exit_m), 
    ('severe', G_severe, start_s, exit_s),
    ('extreme', G_extreme, start_e, exit_e)
]

for name, G, _, _ in graph_cities:
    print(f"{name}: {G.number_of_edges()} edges, {G.number_of_nodes()} nodes")

connected: 60 edges, 26 nodes
moderate: 58 edges, 26 nodes
severe: 56 edges, 26 nodes
extreme: 48 edges, 26 nodes


In [45]:
for name, G, starts, exits in graph_cities:
    for s in starts:
        for e in exits:
            paths = list(nx.all_simple_paths(G, s, e, cutoff=10))
            if s == 'Start_A_1' and e == 'Exit_A_1':
                print(f"{name} | {s} → {e}: {len(paths)} paths")

connected | Start_A_1 → Exit_A_1: 4087 paths
moderate | Start_A_1 → Exit_A_1: 3285 paths
severe | Start_A_1 → Exit_A_1: 2435 paths
extreme | Start_A_1 → Exit_A_1: 273 paths


The path counts are monotonically decreasing exactly as expected amongst the cities.

## Checking PoA

In [46]:
n_experiments = 1
rng = np.random.default_rng(42)
algorithm_seeds = rng.integers(0, 10**6, size=n_experiments)

num_agents = 20
simulation_params = {"congestion": True, "walking": False, "fitness": "max"}

In [47]:
from utils.environment import Environment 
from utils.algorithm_evaluation import AlgorithmComparison
from multi_agent_ga import simulate_ga_evacuation, simulate_greedy_evacuation


environments = [
    Environment(city_name='connected_city'),
    Environment(city_name='moderate_city'),
    Environment(city_name='severe_city'),
    Environment(city_name='extreme_city'),
]

anarchy = []

for city in environments:
    print('\n' + ('- ' * 40) + city.city_name + (' -' * 40) + '\n')
    greedy_solution, greedy_evaluation = simulate_greedy_evacuation(
        num_agents=num_agents,
        simulation_params=simulation_params,
        algorithm_seed=algorithm_seeds[0],
        city=city,
        verbose=True
    )
    
    ga_solution, ga_evaluation = simulate_ga_evacuation(
        city=city,
        num_agents=num_agents,
        population_size=50,
        algorithm_seed=algorithm_seeds[0],
        max_evolutions=50,
        simulation_params=simulation_params,
        verbose=0,
    )
    
    comparison = AlgorithmComparison(greedy_outputs=[greedy_evaluation], ga_outputs=[ga_evaluation])
    price_of_anarchy = comparison.price_of_anarchy(verbose=True)
    anarchy.append(price_of_anarchy)


- - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - connected_city - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - -


===================================================================================== FINAL RESULTS =====================================================================================
Avg path length: 4.00 | Total time: 26.00 | Average time: 21.50 | Exit utilisation: 0.10 | Avg Path Efficiency: 1.00 | Congestion index: 100% | Avg Congestion delay: 17.50

=================================================================================== FINAL RESULTS ===================================================================================
Avg path length: 5.50 | Total time: 9.00 | Average time: 7.10 | Exit utilisation: 0.05 | Avg Path Efficiency: 1.30 | Congestion index: 95% | Avg Congestion delay: 1.60
Price of Anarchy: 2.889 ± 0.000

- - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - 

In [48]:
anarchy

[(np.float64(2.888888888888889), np.float64(0.0)),
 (np.float64(2.3846153846153846), np.float64(0.0)),
 (np.float64(2.4), np.float64(0.0)),
 (np.float64(2.6666666666666665), np.float64(0.0))]

### Why?
In the connected city, greedy agents all pile onto the same short path because there are many routes causing more congestion. In the extreme city, the network is so constrained that even greedy agents are naturally forced to spread out because there simply aren't many path options. So GA's advantage shrinks because greedy has less room to make bad collective decisions.

Even though the PoA isn't increasing as expected the cities still show validation that they have bottleneck information included in them:
* Structural - edge counts decrease
* Topological - available paths decrease monotonically
* Metric - mean average path length increase 

### Running without congestion

In [49]:
from multi_agent_ga import simulate_ga_evacuation, simulate_greedy_evacuation

for city in environments:
    print('\n' + ('- ' * 40) + city.city_name + (' -' * 40) + '\n')
    greedy_solution, greedy_evaluation = simulate_greedy_evacuation(
        num_agents=num_agents,
        simulation_params={"congestion": False, "walking": False, "fitness": "max"},
        algorithm_seed=algorithm_seeds[0],
        city=city,
        verbose=True
    )


- - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - connected_city - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - -


========================================================= FINAL RESULTS =========================================================
Avg path length: 4.00 | Total time: 4.00 | Average time: 4.00 | Exit utilisation: 0.10 | Avg Path Efficiency: 1.00

- - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - moderate_city - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - -


========================================================= FINAL RESULTS =========================================================
Avg path length: 4.00 | Total time: 4.00 | Average time: 4.00 | Exit utilisation: 0.20 | Avg Path Efficiency: 1.00

- - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - severe_city - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - 